# Tutorial 21 — Test Avro compatibility and schema evolution

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
# ruff: noqa
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()), Path.cwd())
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(update={"customers": 200, "accounts": 300, "cards": 240, "devices": 240, "pix_keys": 160, "merchants": 60})
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(update={"population": population, "simulation": simulation, "fraud": fraud})
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})


**Inspect schema, grain, and counts**


In [ ]:
# ruff: noqa
from fraudtwin.contracts import contract_registry
registry = contract_registry()
print("subjects:", registry.subjects)

**Run the core operation**


In [ ]:
# ruff: noqa
contracts = {subject.name: subject.latest for subject in registry.subjects}
print(contracts)

**Measure and interpret the result**


In [ ]:
# ruff: noqa
sample = {"event_id": "evt-1", "amount": 12.5}
compatible = {**sample, "optional_reason": None}
breaking = {"event_id": 1, "amount": "twelve"}
print({"compatible": compatible, "breaking": breaking})

**Exercise a parameter or failure mode**


In [ ]:
# ruff: noqa
checks = {"add_optional_field": "compatible with default", "rename_required_field": "breaking", "type_change": "breaking"}
display(pl.DataFrame([checks]))

**Write a compact artifact and fingerprint**


In [ ]:
# ruff: noqa
assert checks["add_optional_field"] == "compatible with default"
print("Compatibility decisions must be made before producer rollout.")

**Verify invariants and clean up**


In [ ]:
# ruff: noqa
print("Use Schema Registry compatibility endpoints in the optional Docker path; this cell documents the same policy offline.")

**Optional service integration**


In [ ]:
# ruff: noqa
# A compact inspection is more useful than printing an entire run.
print(payments.select([c for c in ("payment_id", "amount", "initiated_at", "payer_account_id") if c in payments.columns]).head(8))
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})


**Review the expected outcome**


In [ ]:
# ruff: noqa
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))


**Next recommended step**


In [ ]:
# ruff: noqa
print("Optional service cell: use Schema Registry compatibility endpoints.")